# OmniCare Clinical Copilot - Notebook 1: Consultation

**Pipeline:** Audio Recording → Whisper Medical ASR → Transcript → MedGemma → SOAP Note

**Prerequisites:** Run `00_setup_and_models.ipynb` first to load models into GPU memory.

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import torch
from datetime import datetime

# Add utils to path
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.encounter_state import new_encounter_id, blank_encounter, save_encounter, update_stage
from utils.prompts import WHISPER_MEDICAL_PROMPT, SOAP_SYSTEM_PROMPT, SOAP_USER_TEMPLATE
from utils.mcp_tools import transcribe_audio, generate_soap_note, parse_soap_sections

print("Imports loaded.")

# Verify models are loaded (from Notebook 0)
try:
    _ = medgemma_model.device
    _ = asr_pipeline
    print(f"MedGemma: loaded on {medgemma_model.device}")
    print(f"Whisper ASR pipeline: ready")
except NameError:
    print("ERROR: Models not loaded. Run Notebook 0 first!")
    raise

## 2. Initialize Patient Encounter

In [ ]:
# Create a new encounter
encounter_id = new_encounter_id()
encounter = blank_encounter(
    encounter_id=encounter_id,
    patient_name="John Smith",  # Replace with actual patient info
    mrn="MRN-2026-001",
    dob="1965-03-15"
)
save_encounter(encounter)
print(f"Created encounter: {encounter_id}")

## 3. Audio Input

Choose one of the following methods:
- **Option A:** Upload a WAV/MP3 file
- **Option B:** Record audio in the browser
- **Option C:** Use mock text (no audio required)

In [ ]:
# === Option A: Upload audio file ===
# from google.colab import files
# uploaded = files.upload()
# audio_path = list(uploaded.keys())[0]

# === Option B: Record audio in browser ===
# (Requires Colab audio recording widget — see cell below)

# === Option C: Use mock text (skip ASR) ===
USE_MOCK_TEXT = True  # Set to False if using real audio
audio_path = None

MOCK_TRANSCRIPT = (
    "Patient presents with a 3-day history of productive cough with yellowish sputum. "
    "Patient also reports low-grade fever peaking at 100.4 degrees Fahrenheit and fatigue. "
    "No known drug allergies. Past medical history significant for type 2 diabetes mellitus "
    "controlled with metformin 500mg twice daily and hypertension managed with lisinopril 10mg daily. "
    "On examination, temperature 99.8 F, blood pressure 138 over 82, heart rate 88 beats per minute, "
    "respiratory rate 18, SpO2 96 percent on room air. Lung auscultation reveals crackles in the "
    "right lower lobe. I will prescribe amoxicillin 500mg three times daily for 7 days. "
    "Ordering a chest X-ray to rule out pneumonia. CBC and BMP labs ordered. "
    "Patient advised to return if symptoms worsen or fever exceeds 101.5 F. "
    "Follow up in one week."
)

print("Audio input configured.")

In [ ]:
# Optional: Browser-based audio recording for Colab
# Uncomment and run this cell to record audio directly

# from IPython.display import HTML, Audio
# from google.colab import output
# from base64 import b64decode
# import soundfile as sf
# import numpy as np
#
# RECORD_HTML = """
# <script>
# let recorder, chunks = [];
# async function startRecording() {
#   const stream = await navigator.mediaDevices.getUserMedia({audio: {sampleRate: 16000, channelCount: 1}});
#   recorder = new MediaRecorder(stream);
#   recorder.ondataavailable = e => chunks.push(e.data);
#   recorder.start();
#   document.getElementById('status').innerText = 'Recording...';
# }
# async function stopRecording() {
#   recorder.stop();
#   recorder.onstop = async () => {
#     const blob = new Blob(chunks, {type: 'audio/webm'});
#     const reader = new FileReader();
#     reader.onload = () => {
#       google.colab.kernel.invokeFunction('save_audio', [reader.result], {});
#       document.getElementById('status').innerText = 'Saved!';
#     };
#     reader.readAsDataURL(blob);
#   };
# }
# </script>
# <button onclick="startRecording()">Start Recording</button>
# <button onclick="stopRecording()">Stop Recording</button>
# <p id="status">Ready</p>
# """
# display(HTML(RECORD_HTML))

## 4. Medical Speech-to-Text (Whisper ASR)

In [ ]:
if USE_MOCK_TEXT:
    transcript = MOCK_TRANSCRIPT
    print("Using mock transcript (no audio file).")
else:
    if audio_path is None:
        raise ValueError("No audio file provided. Set audio_path or use mock text.")

    print(f"Transcribing: {audio_path}")
    print(f"Using medical vocabulary prompting for better accuracy...")

    transcript = transcribe_audio(
        audio_path=audio_path,
        asr_pipeline=asr_pipeline,
        medical_prompt=WHISPER_MEDICAL_PROMPT
    )

print("\n" + "="*60)
print("TRANSCRIPT")
print("="*60)
print(transcript)

## 5. Generate SOAP Note (MedGemma)

In [ ]:
print("Generating SOAP note with MedGemma...")
print(f"Transcript length: {len(transcript)} chars, {len(transcript.split('.'))} sentences")

# Check SOAP trigger heuristic (>3 sentences, matching existing omnicare-mcp.js logic)
sentence_count = len(transcript.split('.'))
if sentence_count <= 3:
    print(f"WARNING: Only {sentence_count} sentences detected. SOAP quality may be limited.")
    print("In production, we accumulate more text before generating.")

soap_raw = generate_soap_note(
    transcript=transcript,
    model=medgemma_model,
    processor=medgemma_processor,
    max_new_tokens=1024
)

print("\n" + "="*60)
print("SOAP NOTE (Raw)")
print("="*60)
print(soap_raw)

## 6. Parse SOAP Sections

In [ ]:
soap_sections = parse_soap_sections(soap_raw)

print("Parsed SOAP sections:")
for section, content in soap_sections.items():
    has_content = "YES" if content.strip() else "EMPTY"
    print(f"  {section.upper()}: {has_content} ({len(content)} chars)")

print("\n--- Subjective ---")
print(soap_sections["subjective"][:300])
print("\n--- Objective ---")
print(soap_sections["objective"][:300])
print("\n--- Assessment ---")
print(soap_sections["assessment"][:300])
print("\n--- Plan ---")
print(soap_sections["plan"][:300])

## 7. Save to Encounter State

In [ ]:
encounter = update_stage(encounter_id, "consultation", {
    "audio_file": audio_path,
    "transcript": transcript,
    "soap_note": soap_sections
})

print(f"Encounter {encounter_id} updated.")
print(f"Saved to: /content/encounters/{encounter_id}.json")
print(f"\nConsultation stage complete!")
print(f"Next: Run 02_admission_vitals_fhir.ipynb with encounter_id = '{encounter_id}'")

## 8. (Optional) View Full Encounter JSON

In [ ]:
from utils.encounter_state import load_encounter
import json

enc = load_encounter(encounter_id)
print(json.dumps(enc, indent=2)[:3000])